In [1]:
import retro
import cv2
import numpy as np
import gym
from stable_baselines3 import DQN
from stable_baselines3.common.vec_env import DummyVecEnv

In [2]:
def preprocess_frame(frame):
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)  
    frame = cv2.resize(frame, (84, 84)) 
    frame = frame / 255.0  
    return frame


In [3]:
def get_health(info):
    return info['player1_health'], info['player2_health']

def compute_reward(prev_health, curr_health):
    p1_health, p2_health = curr_health
    prev_p1_health, prev_p2_health = prev_health
    
    reward = (prev_p2_health - p2_health) - (prev_p1_health - p1_health) 
    return reward


In [ ]:



class MortalKombatEnv(gym.Env):
    def __init__(self):
        super(MortalKombatEnv, self).__init__()
        self.env = retro.make(game='MortalKombatII-Genesis')
        self.action_space = self.env.action_space
        self.observation_space = gym.spaces.Box(low=0, high=1, shape=(84, 84, 1), dtype=np.float32)
        self.prev_health = (176, 176)  
        
    def step(self, action):
        obs, _, done, info = self.env.step(action)
        obs = preprocess_frame(obs)
        curr_health = get_health(info)
        reward = compute_reward(self.prev_health, curr_health)
        self.prev_health = curr_health
        return obs, reward, done, info

    def reset(self):
        obs = self.env.reset()
        obs = preprocess_frame(obs)
        self.prev_health = (176, 176)
        return obs

    def render(self, mode='human'):
        self.env.render()

def create_env():
    return MortalKombatEnv()
env = DummyVecEnv([create_env])

# Train using DQN
model = DQN("CnnPolicy", env, verbose=1, learning_rate=0.0001, buffer_size=10000)
model.learn(total_timesteps=100000)

model.save("mortal_kombat_subzero")

env.close()


AttributeError: module 'gym.utils.seeding' has no attribute 'create_seed'

In [ ]:
model = DQN.load("mortal_kombat_subzero")

obs = env.reset()
while True:
    action, _ = model.predict(obs)
    obs, reward, done, info = env.step(action)
    env.render()
    if done:
        obs = env.reset()


FileNotFoundError: [Errno 2] No such file or directory: 'mortal_kombat_subzero.zip'